# 04 — Regression Models on Gene Features

Adapted from [SalvatoreRa/tutorial — linear_regression](https://github.com/SalvatoreRa/tutorial/blob/main/genomic%20series/linear_regression.ipynb) and [logistic_regression](https://github.com/SalvatoreRa/tutorial/blob/main/genomic%20series/logistic_regression.ipynb) (Apache-2.0).

**Purpose:** Fit linear and logistic regression models to the GENIE gene indicator features. Benchmark simple baselines before moving to survival models and XGBoost AFT.

**Inputs:**
- `datasets_analysis_dictionary/merged_genie.xlsx` or processed CSV

**Outputs:**
- Coefficient plots, ROC curves, confusion matrices → `reports/figures/`
- Model metrics → `data/processed/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, roc_auc_score, roc_curve,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH    = os.path.join(PROJECT_ROOT, "datasets_analysis_dictionary", "merged_genie.xlsx")
FIG_DIR      = os.path.join(PROJECT_ROOT, "reports", "figures")
PROC_DIR     = os.path.join(PROJECT_ROOT, "data", "processed")
SPLIT_DIR    = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

In [ ]:
# ── Load and prepare ─────────────────────────────────────────────────
df = pd.read_excel(DATA_PATH)
gene_cols = [c for c in df.columns if c.startswith("G__")]
print(f"Gene columns: {len(gene_cols)}, Total rows: {len(df)}")

# Detect a continuous outcome for linear regression
cont_candidates = ["DFS_MONTHS", "OS_MONTHS", "PFS_MONTHS"]
cont_target = next((c for c in cont_candidates if c in df.columns), None)

# Detect a binary outcome for logistic regression
bin_candidates = ["DFS_STATUS", "DFS_EVENT", "EVENT", "OS_STATUS"]
bin_target = next((c for c in bin_candidates if c in df.columns), None)

print(f"Continuous target: {cont_target}")
print(f"Binary target:     {bin_target}")

In [ ]:
# ── Train / test split ───────────────────────────────────────────────
features = df[gene_cols].copy()
features = features.fillna(0)  # binary indicators: NaN → 0 assumption

idx_train, idx_test = train_test_split(
    features.index, test_size=0.2, random_state=42
)

# Save split definitions for reproducibility
pd.Series(idx_train, name="index").to_csv(
    os.path.join(SPLIT_DIR, "train_indices.csv"), index=False
)
pd.Series(idx_test, name="index").to_csv(
    os.path.join(SPLIT_DIR, "test_indices.csv"), index=False
)
print(f"Train: {len(idx_train)}, Test: {len(idx_test)}")

In [ ]:
# ── Linear Regression (continuous outcome) ───────────────────────────
if cont_target:
    mask = df[cont_target].notna()
    X_lr = features.loc[mask]
    y_lr = df.loc[mask, cont_target]

    X_train, X_test = X_lr.loc[X_lr.index.isin(idx_train)], X_lr.loc[X_lr.index.isin(idx_test)]
    y_train, y_test = y_lr.loc[X_train.index], y_lr.loc[X_test.index]

    lr = LinearRegression().fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    print(f"Linear Regression — MSE: {mse:.3f}, R²: {r2:.3f}")

    # Coefficient bar plot (top 20 by magnitude)
    coef_df = pd.Series(lr.coef_, index=gene_cols).sort_values(key=abs, ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    coef_df.plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"Top 20 Linear Regression Coefficients ({cont_target})")
    ax.set_xlabel("Coefficient")
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "linreg_top_coefficients.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No continuous target found — skipping linear regression.")

In [ ]:
# ── Logistic Regression (binary outcome) ─────────────────────────────
if bin_target:
    mask = df[bin_target].notna()
    X_log = features.loc[mask]
    y_log = df.loc[mask, bin_target].astype(int)

    X_train, X_test = X_log.loc[X_log.index.isin(idx_train)], X_log.loc[X_log.index.isin(idx_test)]
    y_train, y_test = y_log.loc[X_train.index], y_log.loc[X_test.index]

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    logr = LogisticRegression(max_iter=1000, penalty="l2", C=1.0, random_state=42)
    logr.fit(X_train_sc, y_train)
    y_pred  = logr.predict(X_test_sc)
    y_proba = logr.predict_proba(X_test_sc)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f"Logistic Regression — Accuracy: {acc:.3f}, AUC: {auc:.3f}")
    print(classification_report(y_test, y_pred))

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {auc:.3f}")
    axes[0].plot([0, 1], [0, 1], "--", color="grey")
    axes[0].set_xlabel("FPR")
    axes[0].set_ylabel("TPR")
    axes[0].set_title(f"ROC Curve — {bin_target}")
    axes[0].legend()

    # Confusion matrix
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[1], cmap="Blues")
    axes[1].set_title("Confusion Matrix")

    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "logreg_roc_confusion.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No binary target found — skipping logistic regression.")

In [ ]:
# ── Save metrics summary ─────────────────────────────────────────────
metrics = []
if cont_target:
    metrics.append({"model": "LinearRegression", "target": cont_target, "MSE": mse, "R2": r2})
if bin_target:
    metrics.append({"model": "LogisticRegression", "target": bin_target, "Accuracy": acc, "AUC": auc})

if metrics:
    pd.DataFrame(metrics).to_csv(
        os.path.join(PROC_DIR, "baseline_regression_metrics.csv"), index=False
    )
    print("Saved → data/processed/baseline_regression_metrics.csv")